# 尺度與標準化

對應 DLI：01、02。

本課使用小型合成資料或 scikit-learn 內建資料，在 CPU 執行，不需下載原課程資料。依序執行所有儲存格。

[在 Google Colab 執行本課](https://colab.research.google.com/github/Samuel-ncu/AI-Stastic/blob/main/notebooks/04_scaling_normalization.ipynb) · [閱讀完整背景講義](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/04-scaling-normalization.md)

建議順序：先讀概念與手算 → 回答檢核 → 執行程式 → 修改練習。

## 學習目標與先備知識

先會算平均數與標準差。學完能手算 Z-score、區分三種常見縮放，並說明為什麼測試資料沿用訓練參數。



## 1. 為什麼同一個現象換單位會影響模型？

**來源摘要 [S2](https://scikit-learn.org/stable/modules/preprocessing.html)：**一些模型的目標函數、距離或正則化會受特徵尺度影響。StandardScaler 將特徵中心化並依其標準差縮放；此運算不會自動使分布成為常態。

**本教材自編例子：**兩個測量點的溫差是 2 °C、位置差是 1 m。把位置改記成 mm，位置差變 1,000。若直接用未處理的平方距離，位置項由 1 變成 1,000,000；現象沒變，計算卻被單位支配。

這不表示所有模型都需要完全相同的處理。樹模型通常較不受正的線性縮放影響；需要依模型與特徵意義判斷。



## 2. 三種變換不要混在一起

| 方法 | 公式 | 使用資訊 | 自編用途 |
|---|---|---|---|
| 已知固定尺度 | $x'=x/255$ | 已知像素編碼範圍 | 8-bit 灰階像素 |
| Min-Max | $x'=(x-a)/(b-a)$ | 訓練最小值 a、最大值 b | 依樣本範圍縮放 |
| Z-score | $z=(x-m)/s_0$ | 訓練平均 m、標準差 $s_0$ | 以標準差為尺度比較 |

Z-score 的 2 表示高於訓練平均兩個訓練標準差；不是「成績 2 分」，也不是機率 2。若新值超過訓練最大值，Min-Max 結果可以大於 1，並不必然是程式錯誤。



## 3. 逐步手算

本教材自編訓練值是 10、20、30。平均 m=20。若用除以 n 的描述性變異數：

$$s_0=\sqrt{\frac{(10-20)^2+(20-20)^2+(30-20)^2}{3}}
=\sqrt{\frac{200}{3}}\approx8.165$$

三個 Z-score 約為 −1.225、0、1.225。新測試值 40 仍使用 m=20 與 $s_0=8.165$，得到約 2.449。**測試結果的平均不必是 0。**

StandardScaler 的縮放標準差使用 ddof=0。這與第一課估計母體變異數使用 n−1 的目的不同；不要因為名稱都叫標準差，就把兩個數值當成同一個。



## 4. 讀懂 fit 與 transform

- fit：從訓練資料學到 m、$s_0$ 等參數。
- transform：使用已學到的參數轉換資料。
- fit_transform：在同一批資料擬合並轉換，通常用於訓練集。

本教材的安全流程是「先切分 → 對訓練集 fit → 對各集合 transform」。不要對測試集重新 fit，否則模型看到的是不同座標系。



## 5. 常見誤解與 DLI 連結

除以 255 對 8-bit 像素有明確意義，但不是所有影像、感測器或預訓練模型都用這個規則。DLI 遷移學習應依所選預訓練模型的輸入規格，確保訓練與推論流程一致。

常數特徵標準差為 0，手寫公式會除以 0，需另外處理。離群值也會拉動平均與標準差；先確認來源，而非為了好看就直接刪除。

## 自我檢核

1. 用上面的訓練參數，測試值 20 的 Z-score 是多少？
2. 原本偏斜的分布做 Z-score 後，一定會變成鐘形嗎？
3. 訓練最小值 10、最大值 30，測試值 40 的 Min-Max 值是多少？

### 參考解答

1. 0。
2. 不會；平移與正的等比例縮放不消除偏斜。
3. (40−10)/(30−10)=1.5；不必然有錯。

## 環境

若套件尚未安裝，先執行下一格。安裝需網路，之後的實驗不需下載資料。

In [ ]:
%pip -q install numpy matplotlib scikit-learn

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
train = np.array([[1, 1000], [2, 2000], [3, 3000]], dtype=float)
test = np.array([[4, 4000]], dtype=float)
scaler = StandardScaler().fit(train)
z_train, z_test = scaler.transform(train), scaler.transform(test)
print('Training mean:', scaler.mean_)
print('Scaled training:', z_train)
print('Scaled test:', z_test)
pixels = np.array([0, 128, 255], dtype=float)
print('Pixels / 255:', pixels / 255)
assert np.allclose(z_train.mean(axis=0), 0)
assert np.allclose(z_train.std(axis=0), 1)

## 如何解讀

訓練集的各特徵轉換後平均約為 0、標準差為 1；測試集不必符合這些數值。StandardScaler 使用 ddof=0 的標準差，與樣本標準差的教學公式有所不同。

## 練習

加入離群值重新擬合，觀察轉換參數。說明為何不能對每個測試批次重新擬合。

提交修改後的程式、結果圖表及 200 字解讀，列出至少一項限制。若用 AI 協助，說明你如何驗證結果。

## 連結原課程

回到 [DLI Notebook](https://github.com/NVDLI/fundamentals-of-deep-learning-zh/tree/main/course_content/tutorials)，找出本概念對應的資料處理、模型訓練或評估步驟。

## 出處與閱讀建議

- **S2**：[scikit-learn，Preprocessing data](https://scikit-learn.org/stable/modules/preprocessing.html)。

本課以所列來源的概念作繁體中文摘要，並加入自行編寫的情境、手算、程式及練習；不是原網站的逐段翻譯。標為「來源摘要」者為改寫，標為「自編」者為本教材教學設計；短引文另以引號及「中譯」標明。來源連結可供對照原文，查閱日期：2026-11-17。
